## Trạng thái bản lưu trên GitHub

Thử nghiệm sensitivity-first trước sửa kỹ thuật; không đại diện kết quả cuối.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

# 04b — GMDCSA24 ME-STGCN-TA Sensitivity-First

Notebook độc lập hoàn toàn. Chỉ cần gắn output của notebook `01-gmdcsa24-data-preparation`, bật GPU T4 và chạy từ trên xuống.

- Giữ kiến trúc ME-STGCN-TA: input 8 kênh, 4 block ST-GCN và temporal attention.
- Chỉ điều chỉnh trọng số lớp Fall và decision threshold bằng inner validation.
- Outer-test subject chỉ được đánh giá đúng một lần.
- Mục tiêu ứng dụng: sensitivity tối thiểu 0.95, sau đó giữ specificity cao nhất có thể.

In [ ]:
from pathlib import Path
import json
import random
import re
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
)
from IPython.display import display

SEED = 42
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 120
PATIENCE = 15
DROPOUT = 0.30

FALL_MULTIPLIERS = [1.0, 1.5, 2.0, 3.0, 4.0]
THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
TARGET_SENSITIVITY = 0.95
MIN_SPECIFICITY = 0.60

CHANNEL_NAMES = [
    "relative_x", "relative_y", "visibility",
    "joint_velocity_x", "joint_velocity_y",
    "torso_angle", "hip_displacement_x", "hip_displacement_y",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/04b_me_stgcn_ta_sensitivity")
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Fall multipliers:", FALL_MULTIPLIERS)
print("Thresholds:", THRESHOLDS)

## 1. Đọc dữ liệu

Trong Kaggle, chọn **Add Input → Notebook Output** và gắn version đã lưu output của notebook 01.

In [ ]:
def find_kaggle_file(filename):
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}. Hãy Add Input output của notebook 01."
        )
    return sorted(candidates, key=lambda p: (len(p.parts), str(p)))[0]

FEATURE_PATH = find_kaggle_file("gmdcsa24_features_32.npz")
POSE_PATH = find_kaggle_file("gmdcsa24_pose_32.npz")
MANIFEST_PATH = find_kaggle_file("gmdcsa24_pose_manifest.csv")
FOLDS_PATH = find_kaggle_file("gmdcsa24_cross_subject_folds.csv")

feature_data = np.load(FEATURE_PATH, allow_pickle=False)
pose_data = np.load(POSE_PATH, allow_pickle=False)
manifest = pd.read_csv(MANIFEST_PATH)
saved_folds = pd.read_csv(FOLDS_PATH)

print("Feature:", FEATURE_PATH)
print("Pose:", POSE_PATH)
print("Manifest:", MANIFEST_PATH)
print("Folds:", FOLDS_PATH)
print("Feature arrays:", feature_data.files)
print("Pose arrays:", pose_data.files)

## 2. Ghép tensor đầu vào 8 kênh

Thứ tự kênh phải được giữ nguyên khi export model và triển khai web.

In [ ]:
def first_available_array(name):
    if name in feature_data.files:
        return feature_data[name]
    if name in pose_data.files:
        return pose_data[name]
    raise KeyError(f"Không tìm thấy mảng {name}")

def normalize_labels(values):
    values = np.asarray(values)
    if values.dtype.kind in "iufb":
        labels = values.astype(np.int64)
    else:
        labels = np.array([
            1 if "FALL" in str(value).upper()
            else 0 if "ADL" in str(value).upper()
            else -1
            for value in values
        ], dtype=np.int64)
    if not np.all(np.isin(labels, [0, 1])):
        raise ValueError("Label phải là ADL/0 hoặc FALL/1")
    return labels

def normalize_subjects(values):
    result = []
    for value in np.asarray(values):
        match = re.search(r"(\d+)", str(value))
        if match is None:
            raise ValueError(f"Không đọc được subject: {value}")
        result.append(int(match.group(1)))
    return np.asarray(result, dtype=np.int64)

skeleton = feature_data["skeleton_features"].astype(np.float32)
motion = feature_data["motion_features"].astype(np.float32)
global_features = feature_data["global_features"].astype(np.float32)

y = normalize_labels(first_available_array("labels"))
subjects = normalize_subjects(first_available_array("subjects"))
video_ids = first_available_array("video_ids").astype(str)

# 5 kênh theo từng khớp.
joint_channels = np.stack([
    skeleton[..., 0],
    skeleton[..., 1],
    skeleton[..., 3],
    motion[..., 0],
    motion[..., 1],
], axis=-1)

# Theo cấu trúc global_features của notebook 01:
# 0:2 = vị trí tâm hông; 6 = góc thân.
hip_position = global_features[..., 0:2]
hip_displacement = hip_position - hip_position[:, :1, :]
torso_angle = global_features[..., 6:7]

body_channels = np.concatenate(
    [torso_angle, hip_displacement], axis=-1
)
body_channels = np.repeat(
    body_channels[:, :, None, :], skeleton.shape[2], axis=2
)

# (N,T,V,C) -> (N,C,T,V)
X = np.concatenate([joint_channels, body_channels], axis=-1)
X = np.transpose(X, (0, 3, 1, 2)).astype(np.float32)

assert X.shape == (160, 8, 32, 33), X.shape
assert len(y) == len(subjects) == len(video_ids) == len(X)
assert np.isfinite(X).all()

print("X:", X.shape)
print("ADL:", int((y == 0).sum()), "| FALL:", int((y == 1).sum()))
print("Subjects:", np.unique(subjects, return_counts=True))
print("Finite:", np.isfinite(X).all())
print("Channels:", CHANNEL_NAMES)

## 3. Graph MediaPipe và augmentation

Augmentation chỉ áp dụng cho training loader. Validation và test luôn dùng dữ liệu gốc.

In [ ]:
POSE_EDGES = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),(27,31),(28,32),
]

LEFT_RIGHT_PAIRS = [
    (1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
    (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32),
]

def make_normalized_adjacency(num_nodes=33):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for source, target in POSE_EDGES:
        adjacency[source, target] = 1.0
        adjacency[target, source] = 1.0
    degree = adjacency.sum(axis=1)
    inv_sqrt = np.power(np.maximum(degree, 1e-8), -0.5)
    adjacency = inv_sqrt[:, None] * adjacency * inv_sqrt[None, :]
    return torch.tensor(adjacency, dtype=torch.float32)

ADJACENCY = make_normalized_adjacency()

def augment_sequence(sequence):
    sequence = sequence.clone()
    _, num_frames, _ = sequence.shape

    # Crop 80–100% thời gian rồi nội suy lại 32 frame.
    if random.random() < 0.50:
        crop_length = random.randint(max(8, int(0.80 * num_frames)), num_frames)
        start = random.randint(0, num_frames - crop_length)
        crop = sequence[:, start:start + crop_length, :]
        sequence = F.interpolate(
            crop.permute(2, 0, 1),
            size=num_frames,
            mode="linear",
            align_corners=False,
        ).permute(1, 2, 0)

    # Lật ngang và hoán đổi khớp trái-phải.
    if random.random() < 0.50:
        original = sequence.clone()
        for left, right in LEFT_RIGHT_PAIRS:
            sequence[:, :, left] = original[:, :, right]
            sequence[:, :, right] = original[:, :, left]
        sequence[[0, 3, 5, 6]] *= -1.0

    # Nhiễu nhẹ trên tọa độ x, y.
    if random.random() < 0.35:
        sequence[0:2] += torch.randn_like(sequence[0:2]) * 0.01

    # Bỏ ngẫu nhiên 1–3 joint.
    if random.random() < 0.20:
        joint_ids = torch.randperm(sequence.shape[2])[:random.randint(1, 3)]
        sequence[:, :, joint_ids] = 0.0

    return sequence

class PoseSequenceDataset(Dataset):
    def __init__(self, features, labels, augment=False):
        self.features = torch.from_numpy(features).float()
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        sequence = self.features[index]
        if self.augment:
            sequence = augment_sequence(sequence)
        return sequence, self.labels[index]

def make_loader(indices, augment, shuffle, seed):
    dataset = PoseSequenceDataset(X[indices], y[indices], augment=augment)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )

print("Adjacency:", tuple(ADJACENCY.shape))
print("Số cạnh:", len(POSE_EDGES))

## 4. Mô hình ME-STGCN-TA

Kiến trúc một luồng: `8→64→64→128→128`, spatial average và temporal attention pooling.

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.register_buffer("adjacency", adjacency.clone())
        self.edge_importance = nn.Parameter(torch.ones_like(adjacency))

    def forward(self, x):
        x = self.projection(x)
        graph = self.adjacency * self.edge_importance
        return torch.einsum("nctv,vw->nctw", x, graph)

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.0):
        super().__init__()
        self.graph_conv = SpatialGraphConv(in_channels, out_channels, adjacency)
        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )
        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.temporal_conv(self.graph_conv(x)) + self.residual(x))

class MESTGCNTA(nn.Module):
    def __init__(self, adjacency, in_channels=8, num_classes=2, dropout=DROPOUT):
        super().__init__()
        num_nodes = adjacency.shape[0]
        self.data_bn = nn.BatchNorm1d(in_channels * num_nodes)
        self.blocks = nn.Sequential(
            STGCNBlock(in_channels, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),
            STGCNBlock(128, 128, adjacency, stride=1, dropout=dropout),
        )
        self.temporal_attention = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(64, 1, kernel_size=1),
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        batch_size, channels, frames, nodes = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, nodes * channels, frames)
        x = self.data_bn(x)
        x = x.view(batch_size, nodes, channels, frames).permute(0, 2, 3, 1).contiguous()
        x = self.blocks(x)
        temporal_features = x.mean(dim=3)
        attention = torch.softmax(self.temporal_attention(temporal_features), dim=2)
        video_features = (temporal_features * attention).sum(dim=2)
        logits = self.classifier(video_features)
        return logits, attention.squeeze(1)

model_check = MESTGCNTA(ADJACENCY).to(DEVICE)
with torch.no_grad():
    logits_check, attention_check = model_check(torch.zeros(2, 8, 32, 33, device=DEVICE))
print("Logits:", tuple(logits_check.shape))
print("Attention:", tuple(attention_check.shape))
print("Số tham số:", f"{sum(p.numel() for p in model_check.parameters()):,}")
del model_check
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Metric và chọn operating point

Không chọn tham số bằng outer-test. Inner validation ưu tiên sensitivity ≥ 0.95, specificity ≥ 0.60, sau đó specificity và F2.

In [ ]:
def metrics_from_predictions(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "false_alarm_rate": 1.0 - specificity,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def metrics_at_threshold(y_true, probabilities, threshold):
    result = metrics_from_predictions(y_true, (probabilities >= threshold).astype(np.int64))
    result["threshold"] = float(threshold)
    return result

def operating_rank(metrics):
    sensitivity = metrics["sensitivity"]
    specificity = metrics["specificity"]
    f2 = metrics["f2"]
    if sensitivity >= TARGET_SENSITIVITY and specificity >= MIN_SPECIFICITY:
        return (2, specificity, f2, sensitivity)
    if sensitivity >= TARGET_SENSITIVITY:
        return (1, specificity, f2, sensitivity)
    return (0, sensitivity, specificity, f2)

def choose_threshold(y_true, probabilities):
    candidates = []
    for threshold in THRESHOLDS:
        result = metrics_at_threshold(y_true, probabilities, threshold)
        result["rank"] = operating_rank(result)
        candidates.append(result)
    return max(candidates, key=lambda item: item["rank"])

print("Quy tắc threshold: OK")

## 6. Hàm huấn luyện

Trọng số lớp được tính riêng từ train split; lớp Fall sau đó được nhân thêm hệ số đang thử.

In [ ]:
def make_class_weights(train_indices, fall_multiplier):
    counts = np.bincount(y[train_indices], minlength=2).astype(np.float32)
    if np.any(counts == 0):
        raise ValueError(f"Train split thiếu lớp: {counts}")
    weights = len(train_indices) / (2.0 * counts)
    weights[1] *= fall_multiplier
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_items = 0
    for features, labels in loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_items += len(labels)
    return total_loss / max(total_items, 1)

@torch.no_grad()
def predict_probabilities(model, loader):
    model.eval()
    labels_all, probabilities_all, attention_all = [], [], []
    for features, labels in loader:
        logits, attention = model(features.to(DEVICE, non_blocking=True))
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        labels_all.append(labels.numpy())
        probabilities_all.append(probabilities.cpu().numpy())
        attention_all.append(attention.cpu().numpy())
    return (
        np.concatenate(labels_all),
        np.concatenate(probabilities_all),
        np.concatenate(attention_all),
    )

def fit_inner_fold(train_indices, val_indices, fall_multiplier, run_seed):
    set_seed(run_seed)
    train_loader = make_loader(train_indices, True, True, run_seed)
    val_loader = make_loader(val_indices, False, False, run_seed)
    model = MESTGCNTA(ADJACENCY).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=make_class_weights(train_indices, fall_multiplier))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_state = None
    best_epoch = 0
    best_choice = None
    best_probabilities = None
    stale_epochs = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_one_epoch(model, train_loader, criterion, optimizer)
        val_labels, val_probabilities, _ = predict_probabilities(model, val_loader)
        choice = choose_threshold(val_labels, val_probabilities)

        if best_choice is None or choice["rank"] > best_choice["rank"]:
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            best_epoch = epoch
            best_choice = choice
            best_probabilities = val_probabilities.copy()
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_choice": best_choice,
        "val_probabilities": best_probabilities,
        "epochs_ran": epoch,
    }

def fit_fixed_epochs(train_indices, fall_multiplier, num_epochs, run_seed):
    set_seed(run_seed)
    loader = make_loader(train_indices, True, True, run_seed)
    model = MESTGCNTA(ADJACENCY).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=make_class_weights(train_indices, fall_multiplier))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    for _ in range(num_epochs):
        train_one_epoch(model, loader, criterion, optimizer)
    return model

print("Hàm train: OK")

## 7. Nested LOSO sensitivity-first

Cell này chỉ định nghĩa quy trình. Cell kế tiếp mới bắt đầu train.

In [ ]:
def run_nested_loso_sensitivity():
    unique_subjects = sorted(np.unique(subjects).tolist())
    num_samples = len(y)
    oof_probabilities = np.full(num_samples, np.nan, dtype=np.float32)
    oof_predictions = np.full(num_samples, -1, dtype=np.int64)
    oof_thresholds = np.full(num_samples, np.nan, dtype=np.float32)
    oof_multipliers = np.full(num_samples, np.nan, dtype=np.float32)
    outer_records, inner_records, search_records = [], [], []
    start_time = time.time()

    for outer_number, test_subject in enumerate(unique_subjects, start=1):
        print("\n" + "=" * 76)
        print(f"OUTER {outer_number}/{len(unique_subjects)} | Test Subject {test_subject}")
        test_indices = np.where(subjects == test_subject)[0]
        outer_train_indices = np.where(subjects != test_subject)[0]
        inner_subjects = [subject for subject in unique_subjects if subject != test_subject]
        best_candidate = None

        for multiplier_number, fall_multiplier in enumerate(FALL_MULTIPLIERS):
            pooled_labels, pooled_probabilities, best_epochs = [], [], []
            print(f"  Multiplier {fall_multiplier:.1f}", end="")

            for inner_number, val_subject in enumerate(inner_subjects):
                val_indices = np.where(subjects == val_subject)[0]
                train_indices = np.where((subjects != test_subject) & (subjects != val_subject))[0]
                run_seed = SEED + outer_number * 10000 + multiplier_number * 1000 + inner_number * 100
                result = fit_inner_fold(train_indices, val_indices, fall_multiplier, run_seed)

                pooled_labels.append(y[val_indices])
                pooled_probabilities.append(result["val_probabilities"])
                best_epochs.append(result["best_epoch"])
                metric = result["best_choice"]
                inner_records.append({
                    "outer_test_subject": test_subject,
                    "inner_val_subject": val_subject,
                    "fall_multiplier": fall_multiplier,
                    "best_epoch": result["best_epoch"],
                    "epochs_ran": result["epochs_ran"],
                    "threshold_at_best_epoch": metric["threshold"],
                    "sensitivity": metric["sensitivity"],
                    "specificity": metric["specificity"],
                    "f2": metric["f2"],
                })
                del result["model"]
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            pooled_labels = np.concatenate(pooled_labels)
            pooled_probabilities = np.concatenate(pooled_probabilities)
            pooled_choice = choose_threshold(pooled_labels, pooled_probabilities)
            final_epochs = max(1, int(np.median(best_epochs)))
            candidate = {
                "fall_multiplier": fall_multiplier,
                "threshold": pooled_choice["threshold"],
                "metrics": pooled_choice,
                "rank": pooled_choice["rank"],
                "best_epochs": best_epochs,
                "final_epochs": final_epochs,
            }
            search_records.append({
                "outer_test_subject": test_subject,
                "fall_multiplier": fall_multiplier,
                "selected_threshold": pooled_choice["threshold"],
                "median_epoch": final_epochs,
                "inner_sensitivity": pooled_choice["sensitivity"],
                "inner_specificity": pooled_choice["specificity"],
                "inner_f2": pooled_choice["f2"],
                "target_reached": pooled_choice["sensitivity"] >= TARGET_SENSITIVITY,
            })
            if best_candidate is None or candidate["rank"] > best_candidate["rank"]:
                best_candidate = candidate

            print(
                f" | Sens={pooled_choice['sensitivity']:.3f}"
                f" Spec={pooled_choice['specificity']:.3f}"
                f" F2={pooled_choice['f2']:.3f}"
                f" threshold={pooled_choice['threshold']:.2f}"
                f" epochs={best_epochs}"
            )

        selected_multiplier = best_candidate["fall_multiplier"]
        selected_threshold = best_candidate["threshold"]
        final_epochs = best_candidate["final_epochs"]
        print(
            f"  -> Chọn multiplier={selected_multiplier:.1f}, "
            f"threshold={selected_threshold:.2f}, epochs={final_epochs}"
        )

        final_seed = SEED + outer_number * 10000 + 9999
        final_model = fit_fixed_epochs(
            outer_train_indices, selected_multiplier, final_epochs, final_seed
        )
        test_loader = make_loader(test_indices, False, False, final_seed)
        test_labels, test_probabilities, test_attention = predict_probabilities(final_model, test_loader)
        test_predictions = (test_probabilities >= selected_threshold).astype(np.int64)
        test_metrics = metrics_from_predictions(test_labels, test_predictions)

        oof_probabilities[test_indices] = test_probabilities
        oof_predictions[test_indices] = test_predictions
        oof_thresholds[test_indices] = selected_threshold
        oof_multipliers[test_indices] = selected_multiplier
        outer_records.append({
            "outer_test_subject": test_subject,
            "fall_multiplier": selected_multiplier,
            "threshold": selected_threshold,
            "final_epochs": final_epochs,
            **test_metrics,
        })

        checkpoint = {
            "model_state_dict": {name: value.detach().cpu() for name, value in final_model.state_dict().items()},
            "adjacency": ADJACENCY.cpu(),
            "channel_names": CHANNEL_NAMES,
            "sequence_length": 32,
            "num_joints": 33,
            "label_mapping": {"ADL": 0, "FALL": 1},
            "outer_test_subject": test_subject,
            "trained_subjects": inner_subjects,
            "fall_multiplier": selected_multiplier,
            "decision_threshold": selected_threshold,
            "epochs": final_epochs,
            "seed": final_seed,
        }
        checkpoint_path = MODEL_DIR / f"outer_test_subject_{test_subject}.pt"
        torch.save(checkpoint, checkpoint_path)

        print(
            f"  Test S{test_subject}: Sens={test_metrics['sensitivity']:.4f} | "
            f"Spec={test_metrics['specificity']:.4f} | "
            f"FN={test_metrics['fn']} | FP={test_metrics['fp']}"
        )
        pd.DataFrame(outer_records).to_csv(OUTPUT_DIR / "outer_partial.csv", index=False)
        pd.DataFrame(inner_records).to_csv(OUTPUT_DIR / "inner_partial.csv", index=False)
        pd.DataFrame(search_records).to_csv(OUTPUT_DIR / "search_partial.csv", index=False)

        del final_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "oof_probabilities": oof_probabilities,
        "oof_predictions": oof_predictions,
        "oof_thresholds": oof_thresholds,
        "oof_multipliers": oof_multipliers,
        "outer_records": outer_records,
        "inner_records": inner_records,
        "search_records": search_records,
        "elapsed_seconds": time.time() - start_time,
    }

print("Nested LOSO: sẵn sàng")

## 8. Chạy thí nghiệm

Tối đa 60 inner runs và 4 outer runs. Không đóng phiên Kaggle khi cell đang chạy.

In [ ]:
experiment = run_nested_loso_sensitivity()
print("\nHoàn thành sau:", round(experiment["elapsed_seconds"] / 60, 2), "phút")

## 9. Tổng hợp và lưu kết quả

Sensitivity gộp được tính từ dự đoán outer-test của đủ 160 video. Mỗi outer fold có thể dùng threshold riêng đã chọn bằng inner validation.

In [ ]:
oof_probabilities = experiment["oof_probabilities"]
oof_predictions = experiment["oof_predictions"]
oof_thresholds = experiment["oof_thresholds"]
oof_multipliers = experiment["oof_multipliers"]
outer_df = pd.DataFrame(experiment["outer_records"])
inner_df = pd.DataFrame(experiment["inner_records"])
search_df = pd.DataFrame(experiment["search_records"])

if np.isnan(oof_probabilities).any() or np.any(oof_predictions < 0):
    raise RuntimeError("Chưa có dự đoán outer-test cho đủ 160 video")
if len(outer_df) != 4:
    raise RuntimeError(f"Cần 4 outer folds, hiện có {len(outer_df)}")
if len(inner_df) != 4 * 3 * len(FALL_MULTIPLIERS):
    raise RuntimeError("Số inner runs không đúng")

pooled_metrics = metrics_from_predictions(y, oof_predictions)
prediction_df = pd.DataFrame({
    "sample_index": np.arange(len(y)),
    "video_id": video_ids,
    "subject": subjects,
    "label": y,
    "fall_probability": oof_probabilities,
    "threshold": oof_thresholds,
    "fall_multiplier": oof_multipliers,
    "prediction": oof_predictions,
})
prediction_df["correct"] = prediction_df["label"] == prediction_df["prediction"]

outer_df.to_csv(OUTPUT_DIR / "outer_fold_results.csv", index=False)
inner_df.to_csv(OUTPUT_DIR / "inner_fold_results.csv", index=False)
search_df.to_csv(OUTPUT_DIR / "inner_search_summary.csv", index=False)
prediction_df.to_csv(OUTPUT_DIR / "outer_test_predictions_160.csv", index=False)

summary = {
    "experiment": "04b-gmdcsa24-me-stgcn-ta-sensitivity",
    "protocol": "Nested LOSO safety-oriented operating point",
    "target_sensitivity": TARGET_SENSITIVITY,
    "minimum_specificity": MIN_SPECIFICITY,
    "fall_multiplier_candidates": FALL_MULTIPLIERS,
    "threshold_candidates": THRESHOLDS.tolist(),
    "pooled_metrics": {
        key: float(value) if isinstance(value, (float, np.floating)) else int(value)
        for key, value in pooled_metrics.items()
    },
    "elapsed_seconds": float(experiment["elapsed_seconds"]),
}
with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("KẾT QUẢ GỘP 160 VIDEO")
for name in [
    "accuracy", "balanced_accuracy", "precision", "sensitivity",
    "specificity", "f1", "f2", "false_alarm_rate",
]:
    print(f"{name}: {pooled_metrics[name]:.4f}")
print(
    f"Confusion matrix: TN={pooled_metrics['tn']}, FP={pooled_metrics['fp']}, "
    f"FN={pooled_metrics['fn']}, TP={pooled_metrics['tp']}"
)

if pooled_metrics["sensitivity"] >= TARGET_SENSITIVITY:
    print("\nĐẠT mục tiêu sensitivity >= 95% trên outer-test gộp")
else:
    print("\nCHƯA ĐẠT sensitivity 95% trên subject chưa thấy")
    print("Không tiếp tục chỉnh bằng chính outer-test này")

print("\nKẾT QUẢ TỪNG SUBJECT")
display(outer_df[[
    "outer_test_subject", "fall_multiplier", "threshold", "final_epochs",
    "accuracy", "sensitivity", "specificity", "f2", "fp", "fn",
]])

print("\nTRUNG BÌNH ± ĐỘ LỆCH CHUẨN THEO SUBJECT")
for name in [
    "accuracy", "balanced_accuracy", "precision", "sensitivity",
    "specificity", "f1", "f2", "false_alarm_rate",
]:
    print(f"{name}: {outer_df[name].mean():.4f} ± {outer_df[name].std(ddof=0):.4f}")

archive_path = shutil.make_archive(
    "/kaggle/working/04b_me_stgcn_ta_sensitivity_results",
    "zip",
    OUTPUT_DIR,
)
print("\nĐã lưu thư mục:", OUTPUT_DIR)
print("File tải về:", archive_path)

## 10. Phân tích lỗi

Chỉ dùng để mô tả kết quả sau khi đánh giá; không tiếp tục điều chỉnh bằng các outer-test lỗi.

In [ ]:
false_negatives = prediction_df[
    (prediction_df["label"] == 1) & (prediction_df["prediction"] == 0)
].sort_values("fall_probability")

false_positives = prediction_df[
    (prediction_df["label"] == 0) & (prediction_df["prediction"] == 1)
].sort_values("fall_probability", ascending=False)

print("Fall bị bỏ sót:", len(false_negatives))
display(false_negatives)
print("ADL bị báo nhầm:", len(false_positives))
display(false_positives)

## Lưu version

Sau khi chạy xong:

1. Chọn **Save Version → Quick Save**.
2. Version name: `v1-me-stgcn-ta-sensitivity-nested-loso`.
3. Bật **Always save output when creating a Quick Save**.
4. Tải file `04b_me_stgcn_ta_sensitivity_results.zip`.
5. Gửi lại bảng kết quả gộp và từng subject.

Các checkpoint hiện tại phục vụ đánh giá outer-fold. Sau khi khóa kết quả 04b, notebook tiếp theo sẽ train một model final trên toàn bộ 160 video và export ONNX cho web.